### RAG Pipeline - Data Ingestion to Vector DB

In [8]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
# Read all pdf inside the directory

def process_all_pdfs(pdf_directory):
    all_docs = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory.")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_type"] = "PDF"

            all_docs.extend(documents)
            print(f"Loaded {len(documents)} Pages")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
        
    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs

all_documents = process_all_pdfs("../data/pdf")

Found 3 PDF files in the directory.

Processing ..\data\pdf\docker.pdf...
Loaded 4 Pages

Processing ..\data\pdf\fastapi.pdf...
Loaded 4 Pages

Processing ..\data\pdf\python.pdf...
Loaded 4 Pages

Total documents loaded: 12


In [11]:
all_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-18T09:48:57+00:00', 'source': 'docker.pdf', 'file_path': '..\\data\\pdf\\docker.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'Introduction to Docker', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-09-18T09:48:57+00:00', 'trapped': '', 'modDate': "D:20260918094857+00'00'", 'creationDate': "D:20260918094857+00'00'", 'page': 0, 'file_type': 'PDF'}, page_content='Introduction to Docker\n1. What is Docker?\nDocker is an open-source platform that allows developers to package applications, along with all of\ntheir dependencies, into standardized units called containers. A container includes everything an\napplication needs to run - code, runtime, system tools, libraries, and configuration files - so it behaves\nthe same way regardless of where it is deployed.\nDocker was first released in 2013 and quickly became the stand

In [12]:
# Splitting the documents into smaller chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [13]:
chunks = split_documents(all_documents)
chunks

Split 12 documents into 22 chunks.
Example chunk:
Content: Introduction to Docker
1. What is Docker?
Docker is an open-source platform that allows developers to package applications, along with all of
their dependencies, into standardized units called contain...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-18T09:48:57+00:00', 'source': 'docker.pdf', 'file_path': '..\\data\\pdf\\docker.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'Introduction to Docker', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-09-18T09:48:57+00:00', 'trapped': '', 'modDate': "D:20260918094857+00'00'", 'creationDate': "D:20260918094857+00'00'", 'page': 0, 'file_type': 'PDF'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-18T09:48:57+00:00', 'source': 'docker.pdf', 'file_path': '..\\data\\pdf\\docker.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'Introduction to Docker', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-09-18T09:48:57+00:00', 'trapped': '', 'modDate': "D:20260918094857+00'00'", 'creationDate': "D:20260918094857+00'00'", 'page': 0, 'file_type': 'PDF'}, page_content='Introduction to Docker\n1. What is Docker?\nDocker is an open-source platform that allows developers to package applications, along with all of\ntheir dependencies, into standardized units called containers. A container includes everything an\napplication needs to run - code, runtime, system tools, libraries, and configuration files - so it behaves\nthe same way regardless of where it is deployed.\nDocker was first released in 2013 and quickly became the stand

### Embedding and VectorStoreDB

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\user\OneDrive\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 108.67it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\user\AppData\Local\Temp\ipykernel_21584\1888311026.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


### Vector Store

In [16]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [17]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 22 texts...


Batches: 100%|██████████| 1/1 [00:13<00:00, 13.47s/it]


Generated embeddings with shape: (22, 384)
Adding 22 documents to vector store...
Successfully added 22 documents to vector store
Total documents in collection: 22


### Retriever Pipeline for Vector Store


In [18]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [19]:
rag_retriever

In [21]:
rag_retriever.retrieve("What is python")

Retrieving documents for query: 'What is python'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.38it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_434c971d_14',
  'content': "Introduction to Python\n1. What is Python?\nPython is a high-level, general-purpose programming language created by Guido van Rossum and\nfirst released in 1991. It emphasizes code readability, using significant indentation and a clean,\nsimple syntax that lets developers express ideas in fewer lines of code than languages such as C++\nor Java. Python is an interpreted language, which means code is executed line by line by the Python\ninterpreter rather than being compiled into machine code ahead of time.\nToday Python is one of the most widely used programming languages in the world. It powers web\napplications, data analysis pipelines, machine learning systems, automation scripts, and much more.\nIts large standard library and huge ecosystem of third-party packages make it a practical choice for\nbeginners and professional developers alike.\n2. Main Features of Python\nG\nEasy to learn and read: Python's syntax closely resembles plain English.

### RAG Pipeline- VectorDB To LLM Output Generation

In [42]:
import os
from dotenv import load_dotenv

load_dotenv()



True

In [25]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [37]:
# Simple RAG
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.1,
    max_tokens=1024,
)


def rag_simple(query, retriever, llm, top_k=3):
    # Retriever context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    # Generate response using LLM
    prompt = f"""Use the following context to answer the question concisely:
    Context: {context}
    Question: {query}
    Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [39]:
answer = rag_simple("What is docker", rag_retriever, llm)
answer

Retrieving documents for query: 'What is docker'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.66it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


'Docker is an open‑source platform that packages applications together with all their dependencies into standardized, portable containers, ensuring they run consistently across any environment.'

### Enhanced RAG

In [41]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Container in docker", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Container in docker'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.69it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: A Docker **container** is a lightweight, portable runtime instance of a Docker **image**. It bundles the application code together with everything it needs to run—runtime, system tools, libraries, and configuration—so the application behaves the same regardless of the host environment. Containers are isolated from one another and the host OS, yet share the host’s kernel, making them fast to start and efficient in resource usage.
Sources: [{'source': 'docker.pdf', 'page': 0, 'score': 0.2101142406463623, 'preview': 'Introduction to Docker\n1. What is Docker?\nDocker is an open-source platform that allows developers to package applications, along with all of\ntheir dependencies, into standardized units called containers. A container includes everything an\napplication needs to run - code, runtime, system tools, libr...'}, {'source': 'docker.pdf', 'page': 1, 'score': 0.16916018724441528, 'preview': 'd

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False,
    ) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(
            question, top_k=top_k, score_threshold=min_score
        )
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc["content"] for doc in results])
            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file", doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get("page", "unknown"),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:120] + "...",
                }
                for doc in results
            ]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i : i + 80], end="", flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke(
                [prompt.format(context=context, question=question)]
            )
            answer = response.content

        # Add citations to answer
        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]
        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer
        )

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append(
            {
                "question": question,
                "answer": answer,
                "sources": sources,
                "summary": summary,
            }
        )

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history,
        }


# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query(
    "what is attention is all you need",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True,
)
print("\nFinal Answer:", result["answer"])
print("Summary:", result["summary"])
print("History:", result["history"][-1])

In [ ]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str = None):
        """
        Initialize Groq LLM

        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError(
                "Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter."
            )

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024,
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context

        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so.""",
        )

        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting

        Args:
            query: User question
            context: Retrieved context

        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"   